In [1]:
"""
Run the maxent_smm pipeline on tiny_dataset.json and save predictions.

For each binary question:
  1. Add a BinaryVariable named 'target' for the question outcome.
  2. Generate context variables via VariableGenerator.
  3. Generate estimates via NaturalEstimateGenerator.
  4. Solve the MaxEnt-SMM distribution.
  5. Read P(Yes) = mean(samples[:, 0]) from the target marginal.
"""

"\nRun the maxent_smm pipeline on tiny_dataset.json and save predictions.\n\nFor each binary question:\n  1. Add a BinaryVariable named 'target' for the question outcome.\n  2. Generate context variables via VariableGenerator.\n  3. Generate estimates via NaturalEstimateGenerator.\n  4. Solve the MaxEnt-SMM distribution.\n  5. Read P(Yes) = mean(samples[:, 0]) from the target marginal.\n"

In [2]:
%load_ext autoreload
%autoreload 2

import datetime
import json
import sys
import numpy as np
import jax
import jax.numpy as jnp
import dotenv

dotenv.load_dotenv()
sys.path.insert(0, "..")

from eval_utils import iter_dataset

In [3]:
with open("data/tiny_dataset.json") as f:
    dataset = json.load(f)

print(f"Loaded {len(dataset)} questions")
dataset[0]

Loaded 24 questions


{'id': '41370',
 'source': 'metaculus',
 'question': 'Will there be at least one podium sweep at the 2026 Winter Olympic Games?',
 'resolution_criteria': 'Resolves to the outcome of the question found at https://www.metaculus.com/questions/41370.',
 'background': 'A podium sweep occurs when athletes from the same country win gold, silver, and bronze in a single Olympic event.\n\nPodium sweeps have become increasingly rare at the Winter Olympics but can occur in disciplines where a country has exceptional depth, such as cross-country skiing, speed skating, alpine skiing, or freestyle skiing.\n\nRecent Winter Olympics illustrate the pattern:\n\n* [2022 Beijing Winter Olympics](https://en.wikipedia.org/wiki/2022_Winter_Olympics#Podium_sweeps): 1 podium sweep (Germany)\n* [2018 PyeongChang Winter Olympics](https://en.wikipedia.org/wiki/2018_Winter_Olympics#Podium_sweeps): 3 podium sweeps (Netherlands, Norway, Germany)\n* [2014 Sochi Winter Olympics](https://en.wikipedia.org/wiki/2014_Winte

In [4]:
from eval_utils import iter_dataset

for d in iter_dataset(dataset):
    print(d["title"])

Will there be at least one podium sweep at the 2026 Winter Olympic Games?
Will Jayden Daniels win the 2025–26 NFL OPOY?
Will there be more 'Riots' in Finland for the 30 days before 2026-02-08T00:00:00Z compared to the 30-day average of 'Riots' over the 360 days preceding 2025-09-09T00:00:00Z?

e.g. If the forecast due date is 2024-01-01 and we have the following data:
Date,'Riots'
2023-11-11,1
2023-10-10,2
to calculate the 30-day average of 'Riots' over the preceding 360 days, we’d have: (1+2)/12=0.25.

In this example, for the question to resolve positively, 1 or more 'Riots' would need to occur in the 30 days leading up to the resolution.
Will there be more than ten times as many 'Protests' in Russia for the 30 days before 2026-02-08T00:00:00Z compared to one plus the 30-day average of 'Protests' over the 360 days preceding 2025-09-09T00:00:00Z?

e.g. If the forecast due date is 2024-01-01 and we have the following data:
Date,'Protests'
2023-11-11,1
2023-10-10,2
to calculate one plus

In [12]:
from calibrated_response.llm.openrouter import OpenRouterClient
from calibrated_response.generation.variable_generator import VariableGenerator
from calibrated_response.generation.natural_estimate_generator import NaturalEstimateGenerator
from calibrated_response.models.variable import BinaryVariable
from calibrated_response.maxent_smm.distribution_builder import DistributionBuilder
from calibrated_response.maxent_smm.maxent_solver import JAXSolverConfig
from calibrated_response.energy_models.feature_weight import FeatureWeightModel

# MODEL = "openai/gpt-4o-mini"
# MODEL = "google/gemini-2.5-pro"
MODEL = "google/gemini-3-flash-preview"


client = OpenRouterClient(model=MODEL)

var_gen = VariableGenerator(client)
est_gen = NaturalEstimateGenerator(client)

config = JAXSolverConfig(
    num_chains=256,
    num_iterations=800,
    mcmc_steps_per_iteration=5,
    learning_rate=5e-4,
    l2_regularization=1e-3,
    hmc_step_size=0.01,
    hmc_leapfrog_steps=10,
    verbose=False,
    continuous_prior="gaussian",
    seed=42,
)

In [13]:
from pathlib import Path
from pydantic import TypeAdapter

from calibrated_response.models.query import EstimateUnion
from calibrated_response.models.variable import BinaryVariable, ContinuousVariable

CACHE_PATH = "caches/archive/llm_cache.json"

_ESTIMATE_ADAPTER = TypeAdapter(EstimateUnion)
_VARIABLE_CLASSES = {"BinaryVariable": BinaryVariable, "ContinuousVariable": ContinuousVariable}


def load_cache() -> dict:
    if Path(CACHE_PATH).exists():
        with open(CACHE_PATH) as f:
            return json.load(f)
    return {}


def save_cache(cache: dict) -> None:
    with open(CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=2)


def _serialize_variables(variables):
    return [{"__class__": type(v).__name__, **v.model_dump()} for v in variables]


def _deserialize_variables(data):
    result = []
    for d in data:
        d = dict(d)
        cls = _VARIABLE_CLASSES.get(d.pop("__class__"), ContinuousVariable)
        result.append(cls.model_validate(d))
    return result


def _serialize_estimates(estimates):
    return [e.model_dump() for e in estimates]


def _deserialize_estimates(data):
    return [_ESTIMATE_ADAPTER.validate_python(d) for d in data]

In [14]:
def predict_binary(question_details, entry_id, cache, n_variables=4, n_estimates=10):
    """Run the maxent_smm pipeline for one binary question entry.

    LLM calls (variable + estimate generation) are cached to CACHE_PATH by
    question id. Re-running with different solver settings will skip the LLM
    and go straight to the solver.

    Note: clear llm_cache.json if you change prediction_date / resolution_date
    in iter_dataset, since those affect the question text used as the prompt.
    """
    # today = datetime.datetime.now().strftime("%Y-%m-%d")

    context = (
        f"Question: {question_details['title']}\n\n"
        f"Background:\n{question_details['description']}\n\n"
        f"Resolution criteria:\n{question_details['resolution_criteria']}\n\n"
        f"Additional context:\n{question_details['fine_print']}"
    )

    if entry_id in cache:
        variables = _deserialize_variables(cache[entry_id]["variables"])
        estimates = _deserialize_estimates(cache[entry_id]["estimates"])
    else:
        # target = BinaryVariable(name="target", description=question_details["title"])
        ctx_vars = var_gen.generate(question=context, n_variables=n_variables)
        variables = list(ctx_vars)
        estimates = est_gen.generate(question=context, variables=variables, num_estimates=n_estimates)

        cache[entry_id] = {
            "variables": _serialize_variables(variables),
            "estimates": _serialize_estimates(estimates),
        }
        save_cache(cache)  # persist after each new entry so progress isn't lost

    builder = DistributionBuilder(variables=variables, estimates=estimates, solver_config=config)

    feature_model = FeatureWeightModel(feature_specs=builder.feature_specs)
    params = feature_model.pack_params(feature_model.init_params(jax.random.PRNGKey(0)))

    solver, info = builder.run_solver(energy_fn=feature_model.energy_fn_flat, init_theta=params)

    # P(Yes) = mean of target marginal in normalised [0,1] space
    samples = info["energy_model"].sample(n_samples=2000)
    p_yes = float(np.mean(samples[:, 0]))

    return p_yes, info

In [15]:
# Run on all questions sequentially.
# Note: JAX recompiles for each unique feature-vector shape (K,), and K varies
# per question depending on how many estimates survive. Expect a compilation
# pause on most questions.
cache = load_cache()
print(f"Cache loaded: {len(cache)} entries already cached")

Cache loaded: 1 entries already cached


In [17]:
# Run on all questions sequentially.
# Note: JAX recompiles for each unique feature-vector shape (K,), and K varies
# per question depending on how many estimates survive. Expect a compilation
# pause on most questions.
cache = load_cache()
print(f"Cache loaded: {len(cache)} entries already cached")

predictions = []
for i, (question_details, entry) in enumerate(zip(iter_dataset(dataset), dataset)):
    cached = entry["id"] in cache
    print(f"[{i+1}/{len(dataset)}] {'(cached) ' if cached else ''}{question_details['title'][:80]}...")
    try:
        p_yes, info = predict_binary(question_details, entry["id"], cache)
        try:
            market_price = float(entry["freeze_datetime_value"])
        except (ValueError, TypeError):
            market_price = None

        predictions.append({
            "id": entry["id"],
            "question": question_details["title"],
            "prediction": p_yes,
            "market_price": market_price,
            "resolved_to": entry["resolved_to"],
            "n_features": info["n_features"],
            "n_estimates": info["n_estimates"],
            "skipped": info["skipped_constraints"],
            "converged": info["converged"],
        })
        print(f"  → {p_yes:.1%}  (features={info['n_features']}, converged={info['converged']})")

    except Exception as e:
        print(f"  ERROR: {e}")
        predictions.append({
            "id": entry["id"],
            "question": question_details["title"],
            "prediction": None,
            "market_price": None,
            "resolved_to": entry["resolved_to"],
            "n_features": None,
            "n_estimates": None,
            "skipped": [],
            "converged": False,
        })

Cache loaded: 0 entries already cached
[1/24] Will there be at least one podium sweep at the 2026 Winter Olympic Games?...
Compiled maxent solver
  → 56.6%  (features=11, converged=False)
[2/24] Will Jayden Daniels win the 2025–26 NFL OPOY?...
Compiled maxent solver
  → 40.0%  (features=10, converged=False)
[3/24] Will there be more 'Riots' in Finland for the 30 days before 2026-02-08T00:00:00...
Compiled maxent solver
  → 57.2%  (features=10, converged=False)
[4/24] Will there be more than ten times as many 'Protests' in Russia for the 30 days b...
Compiled maxent solver
  → 44.9%  (features=10, converged=False)
[5/24] Will there be more 'Battles' in eSwatini for the 30 days before 2026-02-08T00:00...
Compiled maxent solver
  → 43.3%  (features=10, converged=False)
[6/24] Will there be more than ten times as many 'Explosions/Remote violence' in Kenya ...
Compiled maxent solver
  → 41.2%  (features=10, converged=False)
[7/24] Will there be more than ten times as many fatalities in Burk

In [18]:
with open("maxent_smm_predictions.json", "w") as f:
    json.dump(predictions, f, indent=2)

successes = sum(1 for p in predictions if p["prediction"] is not None)
print(f"Saved {successes}/{len(predictions)} successful predictions to maxent_smm_predictions.json")

Saved 24/24 successful predictions to maxent_smm_predictions.json
